# CHB-MIT Seizure Detection — full pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noamshva/Epileptic-Seizure-Classification/blob/main/src/run_pipeline.ipynb)

Encode → analyze → classify, driving the `src/` scripts. Built for Google Colab:
data + outputs live on your Google Drive; code is cloned from GitHub.

## 1 · Setup

In [ ]:
# Install deps (--no-deps for sctn so Colab's numpy/torch/... are NOT downgraded)
!pip -q install mne numba
!pip -q install --no-deps sctn

import os, sys, subprocess
from pathlib import Path

# Clone the code (once).
if not os.path.isdir("/content/Epileptic-Seizure-Classification"):
    !git clone https://github.com/noamshva/Epileptic-Seizure-Classification.git
%cd /content/Epileptic-Seizure-Classification

# sctn's pip wheel ships WITHOUT the resonator parameter files — install the
# vendored copy from the repo into the sctn package so resonator() can find them.
import sctn, shutil
_dst = Path(sctn.__file__).parent / "resonators_params" / "parameters"
_dst.mkdir(parents=True, exist_ok=True)
for _f in Path("sctn_params/resonators_params/parameters").glob("*.json"):
    shutil.copy(_f, _dst)
print("resonator params installed:", len(list(_dst.glob("*.json"))))

from google.colab import drive
drive.mount('/content/drive')

## 2 · Config

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_ROOT = Path("/content/Epileptic-Seizure-Classification")
PY = sys.executable

# Our Drive folder that CONTAINS dataset/chbmit and dataset/spikes_raw:
DATA_ROOT = "/content/drive/MyDrive/Msc/Epileptic-Seizure-Classification"
os.environ["PIPELINE_DATA_ROOT"] = DATA_ROOT      # scripts read data + write outputs here
DATA = Path(DATA_ROOT)

# Analyze one subject (like the paper); encode + classify all encoded patients.
ALL_PATIENTS = sorted(p.name for p in (DATA / "dataset" / "spikes_raw").glob("chb*")
                      if p.is_dir()) or ["chb01"]
os.environ["PIPELINE_SUBJECTS"]      = ",".join(ALL_PATIENTS)
os.environ["PIPELINE_SUBJECT"]       = "chb01"
os.environ["PIPELINE_EDF"]           = "chb01_03.edf"
os.environ["PIPELINE_SEIZURE_MIN"]   = "49"
os.environ["PIPELINE_NONSEIZURE_MIN"]= "4"
CLASSIFY_PATIENTS = ALL_PATIENTS

print("patients:", ALL_PATIENTS)
assert (DATA / "dataset").is_dir(), f"No dataset/ under {DATA_ROOT} — check DATA_ROOT and the Drive mount."


def run_stage(script_args, desc, tail=6000):
    print(f"\n{'='*70}\n{desc}\n{'='*70}", flush=True)
    r = subprocess.run([PY, "-u", *script_args], env=os.environ, text=True,
                       capture_output=True, errors="replace")
    print((r.stdout or "")[-tail:])
    if r.returncode != 0:
        print("---- STDERR ----\n", (r.stderr or "")[-3000:])
        raise RuntimeError(f"{desc} failed (exit {r.returncode})")

def show_figs(*names):
    from IPython.display import Image, display
    for n in names:
        p = DATA / "analysis" / "figures" / n
        if p.is_file():
            print(n); display(Image(filename=str(p)))

## 3 · Encode — all patients (skips already-encoded recordings)

In [ ]:
run_stage(["src/1.signal2spikes_chbmit.py"], "ENCODE")

## 4 · Analyze — SCTN vs FFT for one subject (paper Figs 4/6/7)

In [ ]:
run_stage(["src/2.analyze_chbmit.py"], "ANALYZE")
show_figs("04_fft_vs_sctn_bands.png",
          "08_topography_fft_vs_sctn.png",
          "09_topography_seizure_vs_non_seizure.png")

## 5 · Classify — SVM + Random Forest, all patients

In [ ]:
run_stage(["src/3.run_classify_chbmit.py",  *CLASSIFY_PATIENTS], "SVM")
run_stage(["src/3b.run_classify_rf_chbmit.py", *CLASSIFY_PATIENTS], "Random Forest")

import pandas as pd
from IPython.display import Image, display
res = DATA / "output_data" / "results"
for name in ("svm_classifier_report.csv", "rf_classifier_report.csv"):
    if (res / name).is_file():
        print("\n" + name); display(pd.read_csv(res / name).round(3))
for name in ("svm_confusion_matrix.png", "rf_confusion_matrix.png"):
    if (res / name).is_file():
        display(Image(filename=str(res / name)))